# Gnani Evon v3.3 for Surgeline, on a free Kaggle GPU

Serves a 4-bit community quantization of Evon v3.3 (`luminenio/gnani-evon-v3.3-30B-A3B-GGUF`, Q4_K_M)
with Ollama, and gives Surgeline a public OpenAI-compatible URL through a free Cloudflare quick tunnel.

**Before you run it** (right-hand panel, *Session options*):
- Accelerator: **GPU T4 x2**
- Internet: **On** (needs a phone-verified Kaggle account)

Run the cells top to bottom. Cell 4 downloads the model (the slow part). Cell 6 prints the lines for `backend/.env`.
Keep this tab open while you use it: an idle notebook is stopped. The URL is public and has no key, so stop the
session when you are done.

Not tested end to end by its author: if a cell fails, the note under it says what to try.

In [ ]:
# 1. What we have: two T4s, and enough disk under /tmp for the model
!nvidia-smi -L
!df -h /tmp /kaggle/working | cat

In [ ]:
# 2. Install Ollama (prebuilt, with GPU support: nothing to compile)
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# 3. Start the server in the background. The model stays loaded, with room for the notes prompt.
import os, subprocess, time, requests

MODEL = "hf.co/luminenio/gnani-evon-v3.3-30B-A3B-GGUF:Q4_K_M"
env = dict(os.environ,
           OLLAMA_MODELS="/tmp/ollama_models",     # /kaggle/working is too small for the model
           OLLAMA_HOST="127.0.0.1:11434",
           OLLAMA_KEEP_ALIVE="-1",
           OLLAMA_CONTEXT_LENGTH="8192",
           OLLAMA_NUM_PARALLEL="1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2); print("Ollama is up"); break
    except requests.RequestException:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read()[-2000:])

In [ ]:
# 4. Download Evon (Q4_K_M). Several minutes.
!ollama pull {MODEL}
!ollama list

If the next cell says the model's architecture is unknown, this Ollama build can't run Evon's hybrid
Mamba layers. Use the last cell of this notebook (llama.cpp built from source) instead, then come back to cell 6.

In [ ]:
# 5. One Hindi caller line through the same API Surgeline uses. Shows the speed, and whether the
#    reasoning setting leaves an answer (an empty answer means it spent the tokens thinking).
import json

def ask(effort, max_tokens=800):
    body = {"model": MODEL, "temperature": 0.2, "max_tokens": max_tokens,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": 'Extract facts from a flood rescue call. Return ONLY JSON: '
                                              '{"locality": str|null, "people_count": int|null, "floor": str|null, '
                                              '"water_level": str|null, "vulnerable": [str], "spoken_language": str}'},
                {"role": "user", "content": "हम सैनिकपुरी में हैं, second floor पर, चार लोग हैं, दादी चल नहीं पातीं, पानी कमर तक आ गया है"}]}
    if effort:
        body["reasoning_effort"] = effort
    t0 = time.time()
    r = requests.post("http://127.0.0.1:11434/v1/chat/completions", json=body, timeout=300)
    took = time.time() - t0
    if r.status_code != 200:
        return print(f"reasoning_effort={effort!r}: HTTP {r.status_code} {r.text[:200]}")
    d = r.json()
    print(f"reasoning_effort={effort!r}: {took:.1f} s, {d.get('usage', {}).get('completion_tokens')} tokens out")
    print("   ", (d["choices"][0]["message"].get("content") or "(EMPTY: all tokens went to reasoning)")[:300])

ask(None)        # first call also loads the model into the GPUs: ignore its time
for effort in (None, "low", "none"):
    ask(effort)
!ollama ps

Pick the fastest setting that still gives correct JSON and use it as `LLM_REASONING_EFFORT` (leave it out
if the plain call was best). `ollama ps` should say **100% GPU**. If it says part CPU, replies will be slow:
use Evon for notes only (the second block the next cell prints).

In [ ]:
# 6. A public URL for this server (free Cloudflare quick tunnel, no account), and the .env lines
import re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434",
                           "--http-host-header", "localhost:11434", "--no-autoupdate"],
                          stdout=open("/tmp/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/tmp/tunnel.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
if not url:
    print(open("/tmp/tunnel.log").read()[-2000:])
else:
    print("# --- Evon for everything, Groq behind it: paste into backend/.env ---")
    print(f"LLM_BASE_URL={url}/v1\nLLM_API_KEY=ollama\nLLM_MODEL={MODEL}\nNOTES_LLM_MODEL={MODEL}")
    print("LLM_REASONING_EFFORT=low\nLLM_TIMEOUT_SECONDS=12\nLLM_EXTRACT_MAX_TOKENS=800\nNOTES_LLM_TIMEOUT_SECONDS=90")
    print("LLM_FALLBACK_BASE_URL=https://api.groq.com/openai/v1\nLLM_FALLBACK_API_KEY=<your Groq key>")
    print("LLM_FALLBACK_MODEL=openai/gpt-oss-20b\nLLM_FALLBACK_NOTES_MODEL=openai/gpt-oss-120b")
    print()
    print("# --- or: live call stays on Groq, Evon writes the notes and translations ---")
    print(f"NOTES_LLM_BASE_URL={url}/v1\nNOTES_LLM_API_KEY=ollama\nNOTES_LLM_MODEL={MODEL}\nNOTES_LLM_TIMEOUT_SECONDS=90")

In [ ]:
# 7. Keep the session busy while you test and record. Stop this cell (and the session) when done.
while True:
    alive = server.poll() is None and tunnel.poll() is None
    print(time.strftime("%H:%M:%S"), "serving" if alive else "STOPPED: rerun cells 3 and 6 (the URL will change)", flush=True)
    time.sleep(120)

## Only if cell 5 failed: llama.cpp instead of Ollama

Builds llama.cpp with GPU support (15 minutes or more), stops Ollama, and serves the same model on the same
port. Then run cell 6 for the URL. In `.env`, any `LLM_MODEL` name works with this server; to turn the
model's thinking off use `LLM_EXTRA_BODY={"chat_template_kwargs": {"enable_thinking": false}}`.

In [ ]:
# !apt-get -qq install -y libcurl4-openssl-dev > /dev/null
# !git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
# !cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON > /dev/null
# !cmake --build /tmp/llama.cpp/build --config Release -j 4 --target llama-server | tail -3
# server.terminate()
# server = subprocess.Popen(["/tmp/llama.cpp/build/bin/llama-server",
#                            "-hf", "luminenio/gnani-evon-v3.3-30B-A3B-GGUF:Q4_K_M",
#                            "-ngl", "99", "-c", "8192", "--jinja", "--host", "127.0.0.1", "--port", "11434"],
#                           env=dict(os.environ, LLAMA_CACHE="/tmp/llama_cache"),
#                           stdout=open("/tmp/llama.log", "w"), stderr=subprocess.STDOUT)
# print("loading: watch /tmp/llama.log until it says the server is listening")